
# Notebook 14 · Elasticidad promocional y dependencia del descuento

## Pregunta de negocio
¿Dónde el descuento genera volumen incremental y dónde destruye margen?

## Qué aprenderás
- Cómo comparar antes vs durante promoción.
- Cómo leer revenue uplift junto a margin uplift.
- Cómo detectar dependencia promocional por categoría, store o campaña.

## Outputs
- `nb14_promo_scoreboard.csv`
- `nb14_promo_decisions.csv`


In [1]:

from pathlib import Path
import sys
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 200)

helper_candidates = [
    (Path.cwd() / "../helpers").resolve(),
    (Path.cwd().parent / "helpers").resolve(),
    Path("/mnt/data/FraSoHome_marketing_precision_notebooks/helpers"),
]
helper_path = next((p for p in helper_candidates if p.exists()), None)
if helper_path is None:
    raise FileNotFoundError("No se encontró la carpeta helpers/.")
if str(helper_path) not in sys.path:
    sys.path.append(str(helper_path))

import marketing_precision_utils as mpu

tables = mpu.load_core_tables()
paths = mpu.get_paths()
print("Raíz del caso:", paths["root"])
print("Salida de ficheros:", paths["outputs"])


Raíz del caso: /mnt/data/FraSoHome_case
Salida de ficheros: /mnt/data/FraSoHome_case/notebooks_marketing_precision_outputs


In [2]:

promociones = tables["promociones"].copy()
costes = tables["costes"].copy()
campanas = tables["campanas"].copy()
cupones = tables["cupones"].copy()

results = []
for _, p in promociones.iterrows():
    mask_during = (costes["fecha_transaccion_dt"] >= p["fecha_inicio_dt"]) & (costes["fecha_transaccion_dt"] <= p["fecha_fin_dt"])

    if p["ambito"] == "categoria_tienda" and isinstance(p["scope_id"], str) and "|" in p["scope_id"]:
        cat, store = p["scope_id"].split("|", 1)
        mask_during &= (costes["categoria_principal"].astype(str) == cat) & (costes["store_id"].astype(str) == store)
        mask_before = (
            (costes["fecha_transaccion_dt"] >= p["fecha_inicio_dt"] - pd.Timedelta(days=14))
            & (costes["fecha_transaccion_dt"] < p["fecha_inicio_dt"])
            & (costes["categoria_principal"].astype(str) == cat)
            & (costes["store_id"].astype(str) == store)
        )
    elif p["ambito"] == "categoria":
        cat = p["scope_id"]
        mask_during &= costes["categoria_principal"].astype(str) == str(cat)
        mask_before = (
            (costes["fecha_transaccion_dt"] >= p["fecha_inicio_dt"] - pd.Timedelta(days=14))
            & (costes["fecha_transaccion_dt"] < p["fecha_inicio_dt"])
            & (costes["categoria_principal"].astype(str) == str(cat))
        )
    elif p["ambito"] == "campana":
        mask_during &= costes["campaign_id_atribuida"].astype(str) == str(p["scope_id"])
        mask_before = (
            (costes["fecha_transaccion_dt"] >= p["fecha_inicio_dt"] - pd.Timedelta(days=14))
            & (costes["fecha_transaccion_dt"] < p["fecha_inicio_dt"])
        )
    else:
        continue

    during = costes[mask_during]
    before = costes[mask_before]
    if during.empty or before.empty:
        continue

    before_revenue = before["ingreso_bruto_eur"].sum()
    before_margin = before["margen_contribucion_neto_eur"].sum()
    if before_revenue == 0 or before_margin == 0:
        continue

    results.append({
        "promo_id": p["promo_id"],
        "ambito": p["ambito"],
        "scope_id": p["scope_id"],
        "tipo_promocion": p["tipo_promocion"],
        "descuento_pct": p["descuento_pct"],
        "canal": p["canal"],
        "segmento_objetivo": p["segmento_objetivo"],
        "tx_during": len(during),
        "tx_before": len(before),
        "revenue_uplift_pct": during["ingreso_bruto_eur"].sum() / before_revenue - 1,
        "margin_uplift_pct": during["margen_contribucion_neto_eur"].sum() / before_margin - 1,
        "discount_rate_during": during["descuento_total_eur"].sum() / during["ingreso_bruto_eur"].sum(),
        "discount_rate_before": before["descuento_total_eur"].sum() / before_revenue,
        "margin_efficiency": during["margen_contribucion_neto_eur"].sum() / during["descuento_total_eur"].replace(0, np.nan).sum() if during["descuento_total_eur"].sum() != 0 else np.nan,
    })

promo_scoreboard = pd.DataFrame(results).sort_values(["margin_uplift_pct", "revenue_uplift_pct"], ascending=False)
promo_scoreboard.head(15)


,promo_id,ambito,scope_id,tipo_promocion,descuento_pct,canal,segmento_objetivo,tx_during,tx_before,revenue_uplift_pct,margin_uplift_pct,discount_rate_during,discount_rate_before,margin_efficiency
3,PROMO-0057,categoria,Decoración,descuento_pct,10.0,SMS,VIP_OMNICANAL,11,2,18.504168,47.650733,0.096396,0.063906,4.522076
10,PROMO-0091,categoria,Iluminación,envio_gratis,NaN,WEB,LOCAL_STORE,76,6,11.393918,10.835240,0.080531,0.139430,4.266451
13,PROMO-0098,categoria,Muebles,bundle,NaN,SMS,ACTIVO_BASE,98,13,8.251289,6.486962,0.096741,0.053941,3.190160
11,PROMO-0090,categoria_tienda,Decoración|S003,envio_gratis,NaN,WEB,ACTIVO_BASE,10,1,4.538284,5.550348,0.005894,0.000000,94.422849
7,PROMO-0085,categoria,Iluminación,envio_gratis,NaN,EMAIL,PROMO_SENSIBLE_TEXTIL,44,4,6.627381,5.058497,0.077656,0.062750,4.356133
5,PROMO-0071,categoria,Textil hogar,2a_unidad,NaN,WEB,VIP_OMNICANAL,14,1,5.123691,2.755638,0.116790,0.002765,3.108091
8,PROMO-0084,categoria_tienda,Textil hogar|S002,bundle,NaN,SMS,ACTIVO_BASE,3,1,1.023943,2.341456,0.099295,0.016747,4.679858
1,PROMO-0045,categoria,Iluminación,envio_gratis,NaN,SMS,LOCAL_STORE,13,4,1.063338,1.445172,0.115548,0.118846,2.737611
0,PROMO-0043,categoria,Iluminación,2a_unidad,NaN,SMS,PROMO_SENSIBLE_TEXTIL,11,4,0.561094,0.872154,0.134881,0.118846,2.373319
12,PROMO-0095,categoria_tienda,Textil hogar|S002,descuento_pct,10.0,SMS,PROMO_SENSIBLE_TEXTIL,15,2,0.929505,0.290463,0.082557,0.076154,4.056066


In [3]:

def classify_promo(row):
    if row["margin_uplift_pct"] > 0.10 and row["revenue_uplift_pct"] > 0.05:
        return "Escalar"
    if row["revenue_uplift_pct"] > 0 and row["margin_uplift_pct"] <= 0:
        return "Optimizar mecánica / limitar descuento"
    if row["revenue_uplift_pct"] <= 0 and row["discount_rate_during"] > row["discount_rate_before"]:
        return "Reducir o eliminar"
    return "Mantener bajo control"

promo_scoreboard["decision_recomendada"] = promo_scoreboard.apply(classify_promo, axis=1)

category_dependency = (
    costes.groupby("categoria_principal")
          .agg(
              gross_revenue=("ingreso_bruto_eur", "sum"),
              discount_total=("descuento_total_eur", "sum"),
              net_margin=("margen_contribucion_neto_eur", "sum"),
              tx=("transaction_cost_id", "nunique"),
          )
          .reset_index()
)
category_dependency["discount_rate"] = category_dependency["discount_total"] / category_dependency["gross_revenue"].replace(0, np.nan)
category_dependency["margin_rate"] = category_dependency["net_margin"] / category_dependency["gross_revenue"].replace(0, np.nan)
category_dependency.sort_values("discount_rate", ascending=False)


,categoria_principal,gross_revenue,discount_total,net_margin,tx,discount_rate,margin_rate
3,Textil hogar,115825.38,12918.16,42275.99,272,0.111531,0.364998
0,Decoración,336489.68,34286.01,171036.05,568,0.101893,0.508295
1,Iluminación,247256.39,23915.80,79575.30,369,0.096725,0.321833
2,Muebles,953596.52,78793.36,330032.88,520,0.082628,0.346093
4,iluminación,5777.16,259.81,2686.43,12,0.044972,0.465009
5,muebles,13040.36,218.26,6686.70,9,0.016737,0.512770
6,textil hogar,2455.14,12.03,869.17,9,0.004900,0.354021



## Lectura de negocio

Este análisis no pretende medir causalidad perfecta. Sirve para introducir una disciplina de decisión:

- comparar ventas y margen,
- revisar intensidad de descuento,
- y detectar promociones que “compran” volumen a costa de rentabilidad.

Es un buen punto de partida para un diseño posterior con grupos de control y testing más robusto.


In [4]:

promo_decisions = promo_scoreboard[[
    "promo_id", "ambito", "scope_id", "tipo_promocion", "descuento_pct",
    "revenue_uplift_pct", "margin_uplift_pct", "discount_rate_during", "decision_recomendada"
]].sort_values(["decision_recomendada", "margin_uplift_pct"], ascending=[True, False])

path_board = mpu.export_csv(promo_scoreboard, "nb14_promo_scoreboard.csv")
path_dec = mpu.export_csv(promo_decisions, "nb14_promo_decisions.csv")

print("Guardado:", path_board)
print("Guardado:", path_dec)


Guardado: /mnt/data/FraSoHome_case/notebooks_marketing_precision_outputs/nb14_promo_scoreboard.csv
Guardado: /mnt/data/FraSoHome_case/notebooks_marketing_precision_outputs/nb14_promo_decisions.csv
